# Stable Diffusion 推理过程手动实现（含 DDIM 采样器）

本 Notebook 从零手动实现 Stable Diffusion 文生图的完整推理流程，涵盖：
- **文本编码**：使用 CLIP 将文本 prompt 编码为语义向量
- **去噪网络（U-Net）**：接收带噪 latent 和文本条件，预测噪声
- **DDIM 调度器**：确定性去噪采样，比 DDPM 更快收敛
- **VAE 解码**：将去噪后的 latent 还原为 RGB 图像

## 一、依赖导入与时间步嵌入工具函数

In [ ]:
import torch                              # 导入 PyTorch 深度学习框架，提供张量运算和自动微分能力
import torch.nn as nn                    # 导入神经网络模块，提供 Module、Linear、Conv2d 等核心类
import torch.nn.functional as F          # 导入函数式接口，包含 softmax 等无参数操作
from transformers import CLIPTextModel, CLIPTokenizer  # 从 HuggingFace Transformers 导入 CLIP 文本模型和配套分词器
import math                              # 导入 Python 标准数学库，用于 log、exp 等数学计算


def get_timestep_embedding(timesteps, embedding_dim):
    """
    生成正弦余弦时间步嵌入（Sinusoidal Timestep Embedding）。

    时间步嵌入的作用：让 U-Net 感知当前处于扩散过程的哪个阶段（噪声程度），
    从而针对不同噪声强度采用不同的去噪策略。
    正弦位置编码思路源自 Transformer 论文（Vaswani et al., 2017）。

    参数:
        timesteps (Tensor): 形状为 [B] 的长整型张量，每个元素是批内对应样本的当前时间步索引。
                            - B: 批量大小（batch size），即本次推理的图像数量
                            - 取值范围：[0, T-1]，T 为总扩散步数（如 1000）
        embedding_dim (int): 输出嵌入的特征维度 D，通常与 U-Net 隐藏层通道数一致。

    返回:
        Tensor: 形状为 [B, D] 的浮点嵌入张量。
                - B: 批量大小
                - D: embedding_dim，前 D/2 维为正弦编码，后 D/2 维为余弦编码
    """
    half_dim = embedding_dim // 2                                       # 将嵌入维度一分为二，前半部分用正弦编码，后半部分用余弦编码
    emb = math.log(10000) / (half_dim - 1)                             # 计算频率衰减因子，对应公式中的 log(10000) / (D/2 - 1)，控制频率范围
    emb = torch.exp(torch.arange(half_dim, device=timesteps.device) * -emb)  # 生成 half_dim 个频率值 exp(-k·factor), k∈[0,half_dim-1]；形状 [D/2]，频率从高到低
    emb = timesteps[:, None] * emb[None, :]                            # 广播相乘：timesteps 扩展为 [B,1]，emb 扩展为 [1,D/2]，结果 [B,D/2]
    return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)         # 将正弦和余弦在最后一维拼接，最终形状 [B, D]

## 二、文本编码器（TextEncoder）

使用 OpenAI CLIP 模型的文本分支，将自然语言 prompt 编码为稠密语义向量序列，
作为 U-Net 交叉注意力层的**条件输入（condition）**。

In [ ]:
class TextEncoder:
    """
    文本编码器：基于 OpenAI CLIP 模型，将文本 prompt 编码为语义向量序列。

    CLIP (Contrastive Language-Image Pre-training) 通过对比学习，
    使文本和图像在同一语义空间中对齐，其文本特征可有效引导图像生成过程。

    编码流程：
    1. Tokenizer 将文本转换为 token id 序列（最大长度 77）
    2. CLIPTextModel（Transformer 结构）将 token 序列映射为稠密语义向量
    3. 输出最后一层隐藏状态，形状 [B, L, D]
    """

    def __init__(self, device="cuda"):
        """
        初始化文本编码器。

        参数:
            device (str): 计算设备，"cuda" 使用 GPU，"cpu" 使用 CPU。
        """
        self.tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")       # 加载 CLIP 分词器，词表大小 49408，最大序列长度 77 个 token
        self.text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(device)  # 加载 CLIP 文本 Transformer 编码器并迁移到指定设备
        self.device = device                                                                    # 保存设备标识符，后续张量迁移时使用

    def encode(self, prompt):
        """
        将文本 prompt 编码为语义向量序列。

        参数:
            prompt (str 或 List[str]): 输入的文本提示，可以是单个字符串或字符串列表。

        返回:
            Tensor: 文本嵌入向量序列，形状 [B, L, D]。
                    - B: 批量大小（输入文本条数）
                    - L: token 序列长度（CLIP 最大为 77）
                    - D: 隐藏层维度（ViT-B/32 为 512）
        """
        tokens = self.tokenizer(prompt, return_tensors="pt", padding=True).to(self.device)  # 分词并填充到批内相同长度，返回含 input_ids/attention_mask 的字典，迁移至设备
        with torch.no_grad():                                                                  # 关闭梯度追踪，编码阶段不参与反向传播，节省显存
            text_embeddings = self.text_encoder(**tokens).last_hidden_state                    # 前向推理，取最后一层所有 token 的隐藏状态作为文本表示，形状 [B, L, D]
        return text_embeddings                                                                  # 返回文本嵌入序列，后续作为 CrossAttention 的键（K）和值（V）

## 三、模型定义

### 3.1 交叉注意力（CrossAttention）

实现图像 latent 特征与文本语义特征的融合。
**Q（查询）** 来自图像 latent，**K/V（键/值）** 来自文本嵌入，
使图像中每个空间位置都能关注所有文本 token 的语义信息。

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}}\right)V$$

In [ ]:
class CrossAttention(nn.Module):
    """
    交叉注意力模块（Cross-Attention）：实现图像 latent 与文本语义的交互融合。

    在 Stable Diffusion 的 U-Net 中，交叉注意力允许图像中的每个空间位置
    都能「看到」文本描述中所有 token 的语义信息，从而实现文本对图像生成的引导。
    """

    def __init__(self, latent_dim, context_dim):
        """
        参数:
            latent_dim (int): 图像 latent 特征的通道数 C，作为查询维度和输出维度。
            context_dim (int): 文本嵌入的特征维度 D，作为键/值的输入维度（如 768）。
        """
        super().__init__()                                       # 调用父类 nn.Module 初始化，注册模块的参数
        self.query = nn.Linear(latent_dim, latent_dim)          # 查询投影层：将图像 latent 每个位置特征映射为查询向量 Q，输入输出均为 latent_dim
        self.key = nn.Linear(context_dim, latent_dim)           # 键投影层：将文本嵌入（context_dim）映射到与查询相同的 latent_dim 维度，便于计算点积
        self.value = nn.Linear(context_dim, latent_dim)         # 值投影层：将文本嵌入映射为值向量 V，维度与键相同
        self.out = nn.Linear(latent_dim, latent_dim)            # 输出投影层：对注意力加权后的特征做线性变换，维度不变

    def forward(self, x, context):
        """
        前向传播：计算图像 latent 特征对文本嵌入的交叉注意力。

        参数:
            x (Tensor): 图像 latent 特征图，形状 [B, C, H, W]。
                        - B: 批量大小；C: 通道数（latent_dim）；H,W: 空间高宽
            context (Tensor): 文本嵌入序列，形状 [B, L, D]。
                              - L: token 序列长度；D: 文本嵌入维度（context_dim）

        返回:
            Tensor: 融合了文本信息的图像特征图，形状同输入 [B, C, H, W]。
        """
        B, C, H, W = x.shape                                    # 解包特征图维度：B=批量，C=通道数，H=高度，W=宽度
        x_flat = x.view(B, C, H * W).transpose(1, 2)           # 将特征图展平并转置：[B,C,H,W]→[B,C,HW]→[B,HW,C]，使每个空间位置成为序列中的一个元素
        q = self.query(x_flat)                                   # 计算查询向量 Q：输入 [B,HW,C]，输出 [B,HW,latent_dim]
        k = self.key(context)                                    # 计算键向量 K：输入文本嵌入 [B,L,D]，输出 [B,L,latent_dim]
        v = self.value(context)                                  # 计算值向量 V：形状同 K，为 [B,L,latent_dim]
        attn = torch.matmul(q, k.transpose(-2, -1)) / (C ** 0.5)   # 计算缩放点积注意力分数 Q·Kᵀ/√C：[B,HW,C]·[B,C,L]=[B,HW,L]，表示图像每位置对每个文本 token 的关注度
        attended = torch.matmul(attn, v)                         # 注意力加权聚合：[B,HW,L]·[B,L,C]=[B,HW,C]，每个图像位置得到文本加权融合特征
        out = self.out(attended).transpose(1, 2).view(B, C, H, W)  # 输出投影后还原空间维度：[B,HW,C]→转置→[B,C,HW]→view→[B,C,H,W]
        return out                                               # 返回融合了文本条件的图像特征图，形状 [B, C, H, W]

### 3.2 U-Net 去噪网络

Stable Diffusion 的核心组件，负责根据带噪 latent 和文本条件预测噪声。
采用编码器-解码器对称结构，中间融入时间步嵌入和文本交叉注意力。

In [ ]:
class UNet(nn.Module):
    """
    简化版 U-Net 去噪网络：Stable Diffusion 核心组件，预测各扩散步骤中的噪声。

    关键设计：
    - 时间步嵌入（Time Embedding）：通过 MLP 将时间步信息注入特征，使网络感知当前去噪阶段
    - 交叉注意力（Cross-Attention）：将文本语义条件注入去噪过程，实现文本到图像的引导
    - 编码器-解码器结构：下采样提取特征，上采样还原通道（此简化版保持空间尺寸不变）
    """

    def __init__(self):
        """初始化 U-Net 网络，定义下采样路径、时间步 MLP、交叉注意力层和上采样路径。"""
        super().__init__()                                       # 调用父类初始化，注册所有子模块的参数
        self.down = nn.Sequential(                              # 下采样路径（编码器）：将 4 通道 latent 逐步升维至 128 通道
            nn.Conv2d(4, 64, 3, padding=1),                     # 第一层卷积：[B,4,H,W]→[B,64,H,W]，padding=1 保持空间尺寸不变（same padding）
            nn.ReLU(),                                          # ReLU 激活：将负值置零，引入非线性
            nn.Conv2d(64, 128, 3, padding=1),                   # 第二层卷积：[B,64,H,W]→[B,128,H,W]，特征通道数翻倍
            nn.ReLU()                                           # ReLU 激活
        )
        self.time_mlp = nn.Sequential(                          # 时间步嵌入 MLP：对时间步嵌入向量做非线性变换，使网络感知当前去噪步骤
            nn.Linear(128, 128),                                # 全连接层：输入输出均为 128 维，与特征图通道数一致
            nn.ReLU()                                           # ReLU 激活
        )
        self.cross_attn = CrossAttention(latent_dim=128, context_dim=768)  # 交叉注意力层：图像特征维度 128，文本嵌入维度 768
        self.mid = nn.Sequential(                               # 中间特征提取层：在下采样与上采样之间进一步提炼特征，维度保持不变
            nn.Conv2d(128, 128, 3, padding=1),                  # 卷积：[B,128,H,W]→[B,128,H,W]
            nn.ReLU()                                           # ReLU 激活
        )
        self.up = nn.Sequential(                                # 上采样路径（解码器）：将 128 通道特征降维，还原到与输入 latent 相同的 4 通道
            nn.ConvTranspose2d(128, 64, 3, padding=1),         # 第一层转置卷积：[B,128,H,W]→[B,64,H,W]，padding=1 保持尺寸
            nn.ReLU(),                                          # ReLU 激活
            nn.ConvTranspose2d(64, 4, 3, padding=1)            # 第二层转置卷积：[B,64,H,W]→[B,4,H,W]，输出与输入 latent 通道数匹配
        )

    def forward(self, x, text_emb, timestep):
        """
        前向传播：根据带噪 latent 和文本条件预测当前时间步的噪声。

        参数:
            x (Tensor): 当前时间步的带噪 latent，形状 [B, 4, H, W]。
            text_emb (Tensor): 文本嵌入条件，形状 [B, L, 768]。
            timestep (Tensor): 当前时间步索引，形状 [B]，整数类型。

        返回:
            Tensor: 预测的噪声残差，形状 [B, 4, H, W]，供 DDIM 调度器计算去噪方向。
        """
        x = self.down(x)                                         # 下采样提取特征：[B,4,H,W]→[B,128,H,W]
        B, C, H, W = x.shape                                    # 解包当前特征图维度，C=128
        t_emb = get_timestep_embedding(timestep, C).view(B, C, 1, 1)  # 生成时间步正弦嵌入 [B,128]，reshape 为 [B,C,1,1] 以便广播加到特征图
        x = x + self.time_mlp(t_emb.squeeze(-1).squeeze(-1)).view(B, C, 1, 1)  # 时间步嵌入注入：squeeze 去除末尾两个 1 维→MLP 变换→view 恢复广播形状→加到特征图
        x = self.cross_attn(x, text_emb)                        # 应用文本交叉注意力：将文本条件注入图像特征，输出形状不变 [B,128,H,W]
        x = self.mid(x)                                          # 中间特征提取，形状保持 [B,128,H,W]
        x = self.up(x)                                           # 上采样还原通道数：[B,128,H,W]→[B,4,H,W]
        return x                                                 # 返回预测的噪声张量，形状 [B, 4, H, W]

### 3.3 VAE 解码器（Decoder）

将去噪完成后的 4 通道 latent 还原为 3 通道 RGB 图像。
真实 SD 中对应 `AutoencoderKL.decode`，此处为简化版，保持空间尺寸不变。

In [ ]:
class Decoder(nn.Module):
    """
    VAE 解码器（简化版）：将去噪后的 latent 表示解码为 RGB 图像张量。

    真实 SD 中 VAE 解码器将 4 通道 64×64 latent 解码为 3 通道 512×512 RGB 图像。
    此处简化为仅做通道数变换（4→3），不改变空间尺寸，用于教学演示。
    """

    def __init__(self):
        """初始化解码器网络结构。"""
        super().__init__()                                       # 调用父类初始化
        self.net = nn.Sequential(                               # 解码网络：将 4 通道 latent 解码为 3 通道 RGB 图像
            nn.Conv2d(4, 64, 3, padding=1),                     # 升维卷积：[B,4,H,W]→[B,64,H,W]，提取中间特征表示
            nn.ReLU(),                                          # ReLU 激活，引入非线性
            nn.Conv2d(64, 3, 3, padding=1),                     # 输出卷积：[B,64,H,W]→[B,3,H,W]，对应 RGB 三个颜色通道
            nn.Tanh()                                           # Tanh 激活：将输出约束到 [-1,1]，可通过 (x+1)/2 映射到 [0,1] 像素范围
        )

    def forward(self, x):
        """
        解码前向传播。

        参数:
            x (Tensor): 去噪完成的 latent 张量，形状 [B, 4, H, W]。

        返回:
            Tensor: 生成的 RGB 图像张量，形状 [B, 3, H, W]，像素值范围 [-1, 1]。
        """
        return self.net(x)                                       # 通过解码网络，输出 RGB 图像张量 [B, 3, H, W]

## 四、DDIM 调度器

DDIM（Denoising Diffusion Implicit Models）相比 DDPM 的优势：
- 采样过程**确定性**（deterministic），去除了 DDPM 中每步添加随机噪声的步骤
- 可以用**更少步数**（如 10～50 步）得到与 DDPM 1000 步相当的质量
- 核心采样公式：$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \hat{x}_0 + \sqrt{1-\bar{\alpha}_{t-1}} \varepsilon_\theta$

In [ ]:
class DDIMScheduler:
    """
    DDIM 调度器（简化版）：管理扩散过程的噪声方差序列，并实现确定性去噪采样。

    关键变量说明：
    - β_t (betas)     ：第 t 步添加的噪声方差，从 beta_start 线性增长到 beta_end
    - α_t (alphas)    = 1 - β_t，表示第 t 步原始信号的保留比例
    - ᾱ_t (alpha_bars)= ∏α_i，从步骤 0 到 t 的信号总保留比例（t 越大越接近 0）
    """

    def __init__(self, steps=10, beta_start=0.0001, beta_end=0.02):
        """
        初始化 DDIM 调度器，预计算 beta、alpha 及其累积乘积序列。

        参数:
            steps (int): 总扩散（去噪）步数 T，推理时反向迭代 T 步完成去噪。
            beta_start (float): 噪声方差序列 β 的起始值，控制最小噪声强度。
            beta_end (float): 噪声方差序列 β 的终止值，控制最大噪声强度。
        """
        self.steps = steps                                                          # 保存总步数，用于后续反向迭代循环控制
        self.betas = torch.linspace(beta_start, beta_end, steps)                   # β 序列：从 beta_start 到 beta_end 均匀插值 steps 个值；形状 [steps]，每个 β_t 表示该步的噪声方差
        self.alphas = 1.0 - self.betas                                              # α_t = 1 - β_t：每步保留原始信号的比例，α_t 越接近 1 说明该步噪声越少；形状 [steps]
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)                         # ᾱ_t = ∏α_i：沿 dim=0 计算累积乘积，代表从第 0 步到第 t 步信号总保留比例；形状 [steps]

    def step(self, x, noise_pred, t):
        """
        执行单步 DDIM 去噪，从 x_t 计算前一时间步 x_{t-1}。

        参数:
            x (Tensor): 当前时间步 t 的带噪 latent，形状 [B, C, H, W]。
            noise_pred (Tensor): U-Net 预测的噪声 ε_θ(x_t,t)，形状同 x。
            t (int): 当前时间步整数索引，范围 [0, steps-1]。

        返回:
            Tensor: 去噪一步后的 latent x_{t-1}，形状 [B, C, H, W]。
        """
        alpha_bar = self.alpha_bars[t]                                              # 取当前时间步 t 的 ᾱ_t（标量），代表 t 时刻信号总保留比例
        alpha_bar_prev = self.alpha_bars[max(t - 1, 0)]                             # 取前一时间步 t-1 的 ᾱ_{t-1}；当 t=0 时退化为 ᾱ_0 防止索引越界
        pred_x0 = (x - (1 - alpha_bar).sqrt() * noise_pred) / alpha_bar.sqrt()     # 估计干净图像 x̂₀：从带噪图 x_t 减去预测噪声分量，再除以信号保留比例的开方
        dir_xt = (1 - alpha_bar_prev).sqrt() * noise_pred                          # DDIM 噪声方向项：sqrt(1-ᾱ_{t-1})·ε_θ，决定下一步保留多少噪声方向的分量
        x_prev = alpha_bar_prev.sqrt() * pred_x0 + dir_xt                          # DDIM 核心采样公式：x_{t-1}=sqrt(ᾱ_{t-1})·x̂₀+sqrt(1-ᾱ_{t-1})·ε_θ（确定性，不添加随机噪声）
        return x_prev                                                                # 返回去噪一步后的 latent x_{t-1}，形状 [B, C, H, W]

## 五、完整推理管线与运行示例

将上述所有组件组合为端到端的文生图推理管线，并执行一次完整的图像生成流程。

In [ ]:
class StableDiffusionDemo:
    """
    Stable Diffusion 简化演示管线：整合文本编码器、U-Net、解码器和 DDIM 调度器，
    实现端到端的文本驱动图像生成。

    完整推理流程：
    1. 用 CLIP 文本编码器将 prompt 编码为语义向量 [B, L, D]
    2. 从标准正态分布采样随机 latent（代表纯噪声图像 x_T）
    3. 循环 T 步 DDIM 去噪：U-Net 预测噪声 → 调度器计算 x_{t-1}
    4. 用解码器将去噪后的 latent 解码为 RGB 图像
    """

    def __init__(self, device="cuda"):
        """
        初始化推理管线，实例化所有子模块并迁移到指定设备。

        参数:
            device (str): 计算设备，"cuda" 使用 GPU，"cpu" 使用 CPU。
        """
        self.device = device                                     # 保存设备字符串，统一各子模块的张量存储位置
        self.text_encoder = TextEncoder(device)                 # 实例化 CLIP 文本编码器，负责将 prompt 编码为语义嵌入向量
        self.unet = UNet().to(device)                           # 实例化并迁移 U-Net 去噪网络到计算设备
        self.decoder = Decoder().to(device)                     # 实例化并迁移 VAE 解码器到计算设备
        self.scheduler = DDIMScheduler()                        # 初始化 DDIM 调度器，默认 10 步推理，预计算 alpha 序列

    def generate(self, prompt):
        """
        根据文本 prompt 生成图像张量。

        参数:
            prompt (str): 描述目标图像内容的文本提示，如 "a cat on the moon"。

        返回:
            Tensor: 生成的 RGB 图像张量，形状 [1, 3, 64, 64]，像素值域 [-1, 1]。
                    - 1: 批量大小（单张图像）
                    - 3: RGB 三通道
                    - 64×64: latent 解码后的空间分辨率（简化版，未做空间上采样）
        """
        text_embeddings = self.text_encoder.encode(prompt)     # 将文本 prompt 编码为语义嵌入向量序列，形状 [B, L, D]
        batch_size = text_embeddings.shape[0]                   # B: 批量大小（输入文本条数）
        seq_len = text_embeddings.shape[1]                      # L: token 序列长度（CLIP 最大 77）
        hidden_dim = text_embeddings.shape[2]                   # D: 文本嵌入的隐藏层维度
        if hidden_dim != 768:                                    # 若文本嵌入维度与 CrossAttention 期望的 context_dim(768) 不匹配，则进行线性投影对齐
            projection = nn.Linear(hidden_dim, 768).to(self.device)  # 创建投影层，将任意维度的文本嵌入映射到 768 维
            text_embeddings = projection(text_embeddings)       # 投影后输出形状 [B, L, 768]
        latent = torch.randn(1, 4, 64, 64).to(self.device)     # 在 latent 空间采样纯随机噪声作为初始 x_T；形状 [1,4,64,64]：1张图，4通道，64×64分辨率
        for i in reversed(range(self.scheduler.steps)):         # 从时间步 T-1 逆序迭代到 0，执行 DDIM 反向去噪过程
            t = torch.full((1,), i, device=self.device, dtype=torch.long)  # 构建当前时间步张量，形状 [1]，dtype=long（整数），值为当前步索引 i
            noise_pred = self.unet(latent, text_embeddings, t)  # U-Net 预测当前步的噪声 ε_θ(x_t,t,c)，输入带噪 latent 和文本条件，输出形状 [1,4,64,64]
            latent = self.scheduler.step(latent, noise_pred, i) # 调度器执行一步 DDIM 去噪：根据预测噪声计算 x_{t-1}，逐步从噪声中还原图像
        image = self.decoder(latent)                            # 扩散去噪结束，用解码器将 latent [1,4,64,64] 解码为 RGB 图像 [1,3,64,64]
        return image                                            # 返回最终生成的图像张量，形状 [1, 3, 64, 64]


device = "cuda" if torch.cuda.is_available() else "cpu"         # 自动检测 CUDA 可用性：优先使用 GPU 加速推理，否则退回 CPU
generator = StableDiffusionDemo(device)                         # 实例化 SD 演示管线，完成所有子模块的加载和设备迁移
output = generator.generate("a futuristic city with flying cars")   # 输入英文文本 prompt 执行完整推理，经 DDIM 10 步去噪生成图像
print("图像生成完成，输出张量 shape:", output.shape)              # 打印生成图像张量的形状；期望输出: torch.Size([1, 3, 64, 64])

图像生成完成，输出张量 shape: torch.Size([1, 3, 64, 64])


## 六、`torch.cumprod` 功能演示

`torch.cumprod` 计算张量沿指定维度的**累积乘积**，
在 DDIM 中用于计算 $\bar{\alpha}_t = \prod_{i=1}^{t} \alpha_i$（信号总保留比例）。

In [ ]:
import torch                                                      # 导入 PyTorch 库，提供张量操作和 cumprod 接口

tensor = torch.tensor([0.9, 0.8, 0.7, 0.6])                    # 构造一维浮点张量，模拟 4 个时间步的 alpha 序列（信号保留比例）；形状 [4]

# torch.cumprod(input, dim) 沿 dim 方向依次计算累积乘积：
#   结果[0] = 0.9                     ← 第 0 步的 ᾱ_0
#   结果[1] = 0.9 × 0.8 = 0.72       ← 第 0~1 步的 ᾱ_1
#   结果[2] = 0.72 × 0.7 = 0.504     ← 第 0~2 步的 ᾱ_2
#   结果[3] = 0.504 × 0.6 = 0.3024   ← 第 0~3 步的 ᾱ_3（ᾱ 越来越小，代表噪声越来越多）
cumprod_result = torch.cumprod(tensor, dim=0)                    # 沿第 0 维（唯一的维度）计算累积乘积，对应 DDIM 中的 ᾱ_t 序列；输出形状 [4]

print("原始张量:", tensor)                                      # 打印原始 alpha 序列，形状 [4]，值为 [0.9, 0.8, 0.7, 0.6]
print("累积乘积结果:", cumprod_result)                          # 打印 alpha_bar 累积序列，形状 [4]，值为 [0.9, 0.72, 0.504, 0.3024]

原始张量: tensor([0.9000, 0.8000, 0.7000, 0.6000])
累积乘积结果: tensor([0.9000, 0.7200, 0.5040, 0.3024])
